# 01 · Ajustar YOLO-seg para segmentar las piezas del contenedor

**Para qué.** El demo separa las piezas con un segmentador. Por defecto usa HSV + profundidad (sin entrenar). Este cuaderno ajusta una red
**YOLO de segmentación** con imágenes de la cámara cenital simulada, para compararla con HSV con exactamente las mismas imágenes y etiquetas.
**En el repositorio no había ningún modelo YOLO entrenado**: esto lo crea.

**Entrada.** La carpeta que escribe `tools/export_yolo_dataset.py` en Isaac Sim (`data/yolo_pieces/`), comprimida en un zip y subida a Drive.
Una sola clase, *pieza*: **el color y la forma no los decide la red**, se miden en la máscara (`binpick/attributes.py`), así que la misma red sirve
con otros colores.

**Salida.** `piezas_seg.pt` (mejor modelo), que se usa con `python 18_voice_bin_picking.py --segmenter yolo:piezas_seg.pt` y se evalúa con
`python tools/eval_segmentation.py --data data/yolo_pieces --segmenter yolo:piezas_seg.pt`.

**Tiempo orientativo (sin medir).** ~400 escenas × 3 vistas ≈ 1 200 imágenes; `yolo11n-seg` a 640 px, 100 épocas: del orden de 20–40 min en una T4.
Aviso honesto: las etiquetas son analíticas y la red verá sólo este render; sobre una cámara real habría que volver a ajustarla.

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "· CUDA:", torch.cuda.is_available())

In [ ]:
!pip -q install -U ultralytics
import ultralytics; print("ultralytics", ultralytics.__version__)   # anota la versión junto a los resultados

## Datos
Sube `yolo_pieces.zip` a tu Drive (carpeta `piezas/`) o cámbialo por una subida directa.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ZIP = "/content/drive/MyDrive/piezas/yolo_pieces.zip"     # <- ajusta
!rm -rf /content/yolo && mkdir -p /content/yolo && unzip -q "{ZIP}" -d /content/yolo
import pathlib, glob
root = next(p.parent for p in pathlib.Path("/content/yolo").rglob("data.yaml"))
print("dataset:", root)
for s in ("train", "val"):
    print(s, len(glob.glob(f"{root}/images/{s}/*.jpg")), "imágenes")
print(open(root / "data.yaml").read())

In [ ]:
# Comprobación visual: las etiquetas deben cuadrar con las piezas
import cv2, random, matplotlib.pyplot as plt
imgs = sorted(glob.glob(f"{root}/images/train/*.jpg")); random.seed(0)
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axs, random.sample(imgs, 3)):
    im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lab = f.replace("images", "labels").replace(".jpg", ".txt")
    for line in open(lab):
        v = list(map(float, line.split()[1:])); pts = [(int(v[i] * w), int(v[i + 1] * h)) for i in range(0, len(v), 2)]
        cv2.polylines(im, [__import__("numpy").array(pts, "int32")], True, (255, 255, 0), 1)
    ax.imshow(im); ax.axis("off")
plt.show()

## Entrenar
Valores de partida razonables, no medidos: el recorte de color queda **desactivado** (`hsv_h=0`) porque el color es un atributo que se mide después.

In [ ]:
from ultralytics import YOLO
MODEL, EPOCHS, IMGSZ, BATCH = "yolo11n-seg.pt", 100, 800, 16
model = YOLO(MODEL)
model.train(data=f"{root}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=30, project="/content/runs", name="piezas",
            hsv_h=0.0, hsv_s=0.2, hsv_v=0.3, fliplr=0.5, flipud=0.5, degrees=0, mosaic=1.0, seed=0, workers=4, plots=True)

In [ ]:
best = YOLO("/content/runs/piezas/weights/best.pt")
m = best.val(data=f"{root}/data.yaml", imgsz=IMGSZ, split="val")
print("máscara  mAP50 = %.3f · mAP50-95 = %.3f" % (m.seg.map50, m.seg.map))
print("caja     mAP50 = %.3f · mAP50-95 = %.3f" % (m.box.map50, m.box.map))

## (Opcional) YOLOE sin entrenar, por texto

Útil como punto de comparación y para piezas reales que aún no tienen datos. Necesita descargar pesos (y el codificador de texto la primera vez).
Las métricas **por instancia, color, posición y oclusión** se calculan en local con `tools/eval_segmentation.py` sobre el mismo conjunto; aquí sólo se ve a ojo.

In [ ]:
try:
    from ultralytics import YOLOE
    z = YOLOE("yoloe-11s-seg.pt"); names = ["industrial part", "cube", "cylinder"]
    z.set_classes(names, z.get_text_pe(names))
    r = z.predict(sorted(glob.glob(f"{root}/images/val/*.jpg"))[0], conf=0.25, imgsz=IMGSZ, retina_masks=True, verbose=False)[0]
    print("YOLOE sin entrenar detecta", 0 if r.masks is None else len(r.masks.data), "instancias en la primera imagen de val")
except Exception as e:                       # versión de ultralytics sin YOLOE o sin red
    print("YOLOE no disponible aquí:", e)

In [ ]:
import shutil
shutil.copy("/content/runs/piezas/weights/best.pt", "/content/piezas_seg.pt")
shutil.copy("/content/piezas_seg.pt", "/content/drive/MyDrive/piezas/piezas_seg.pt")
from google.colab import files; files.download("/content/piezas_seg.pt")

## Usarlo en el demo

En la máquina con Isaac Sim (el entorno conda del proyecto necesita `ultralytics` **sin tocar las versiones ya fijadas** de numpy/torch):

```powershell
python -m pip install ultralytics --no-deps
python -m pip install ultralytics-thop polars   # si el import se queja de módulos que faltan; añade lo que pida sin actualizar numpy
python tools/eval_segmentation.py --data data/yolo_pieces --segmenter yolo:piezas_seg.pt
python 18_voice_bin_picking.py --segmenter yolo:piezas_seg.pt --open
```

No se ha probado ninguna instalación de ultralytics en ese entorno: si `import ultralytics` rompe numpy, vuelve a `--segmenter hsv`.